In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics

In [ ]:
import zipfile
import os

zip_path = "(yourpath)/datasetusing.zip"
extract_path = "(yourpath)/braille_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

In [ ]:
import glob

label_files = glob.glob(
    "(yourpath)/braille_dataset/**/*.txt",
    recursive=True
)

for file in label_files:

    if "README" in file:
        continue

    new_lines = []

    with open(file, "r") as f:
        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            parts[0] = "0"
            new_lines.append(" ".join(parts))

    with open(file, "w") as f:
        f.write("\n".join(new_lines))

print("Labels converted to single class")

In [ ]:
yaml_path = "(yourpath)/braille_dataset/data.yaml"

with open(yaml_path, "w") as f:
    f.write("""
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1

names:
  0: braille_cell
""")

print("data.yaml updated")

In [ ]:
!nvidia-smi

In [ ]:
!find (yourpath)/braille_dataset -name "data.yaml"

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="(yourpath)/braille_dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    cache=True,
    workers=4,
    device=0,
    project="(yourpath)/BrailleYOLO",
    name="braille_cell_detector_v2"
)

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

model = YOLO(
    "(yourpath)/best.pt"
)

results = model.predict(
    source="(yourpath)/test.jpeg",
    conf=0.25,
    save=True,
    show_labels=False,
    show_conf=False
)

print("Detections:", len(results[0].boxes))

boxes = results[0].boxes.xyxy.cpu().numpy()

for i, box in enumerate(boxes):
    print(f"Cell {i+1}: {box}")
from IPython.display import Image, display

import os

for root, dirs, files in os.walk("(yourpath)/runs/detect"):
    for file in files:
        if file.endswith((".jpg", ".jpeg", ".png")):
            print(os.path.join(root, file))

from IPython.display import Image, display

display(Image(filename="(yourpath)/test.jpg"))

In [ ]:
import numpy as np

boxes = results[0].boxes.xyxy.cpu().numpy()

# Sort by y-coordinate
boxes = sorted(boxes, key=lambda b: b[1])

rows = []
cell_heights = [b[3] - b[1] for b in boxes]
avg_height = np.mean(cell_heights)

row_threshold = avg_height * 0.7

for box in boxes:
    if not rows:
        rows.append([box])
        continue

    current_y = box[1]
    row_y = rows[-1][0][1]

    if abs(current_y - row_y) < row_threshold:
        rows[-1].append(box)
    else:
        rows.append([box])

# Sort each row left-to-right
for row in rows:
    row.sort(key=lambda b: b[0])

print(f"Rows found: {len(rows)}")

for i, row in enumerate(rows):
    print(f"\nRow {i+1}:")
    for box in row:
        print(f"x={box[0]:.1f}, y={box[1]:.1f}")

In [ ]:
import cv2
import os

img = cv2.imread("(yourpath)/test.jpeg")

os.makedirs("cropped_cells", exist_ok=True)

cell_id = 0

for row in rows:
    for box in row:

        x1, y1, x2, y2 = map(int, box)

        crop = img[y1:y2, x1:x2]

        cv2.imwrite(
            f"cropped_cells/cell_{cell_id:03d}.png",
            crop
        )

        cell_id += 1

print(f"Saved {cell_id} cropped cells")